In [2]:
import numpy as np
from PIL import Image
import onnxruntime as ort

# 定义预处理函数，用于将图片转换为模型所需的输入格式
def preprocess(image_path):
    input_shape = (1, 1, 64, 64)    # 模型输入期望的形状，这里是 (N, C, H, W)，N=batch size, C=channels, H=height, W=width
    img = Image.open(image_path).convert('L')    # 打开图像文件并将其转换为灰度图  1分

    # 【Bug修复】将旧版的 Image.ANTIALIAS 替换为新版兼容的 Image.Resampling.LANCZOS
    img = img.resize((64, 64), Image.Resampling.LANCZOS)

    img_data = np.array(img, dtype=np.float32)    # 将PIL图像对象转换为numpy数组，并确保数据类型是float32

    # 调整数组的形状以匹配模型输入的形状
    img_data = np.expand_dims(img_data, axis=0)  # 添加 batch 维度
    img_data = np.expand_dims(img_data, axis=1)  # 添加 channel 维度

    assert img_data.shape == input_shape, f"Expected shape {input_shape}, but got {img_data.shape}"    # 确保最终的形状与模型输入要求的形状一致
    return img_data    # 返回预处理后的图像数据

emotion_table = {'neutral':0, 'happiness':1, 'surprise':2, 'sadness':3, 'anger':4, 'disgust':5, 'fear':6, 'contempt':7}

ort_session = ort.InferenceSession('emotion-ferplus.onnx')

input_data = preprocess('img_test.png')

ort_inputs = {ort_session.get_inputs()[0].name: input_data}

ort_outs = ort_session.run(None, ort_inputs)

predicted_label = np.argmax(ort_outs[0])

print(predicted_label)

predicted_emotion = list(emotion_table.keys())[predicted_label]

print(predicted_emotion)


2
surprise
